In [1]:
from agent1.analysis.action_plan import ActionPlan, RequestType
from agent1.execution.action_executor import ActionExecutor
from agent1.routing.action_types import ActionType

executor = ActionExecutor()

tracking_plan = ActionPlan(
    request_type=RequestType.SINGLE_ACTION,
    intent="track_order",
    action=ActionType.ORDER_SERVICE,
    reason="The user wants to track an order.",
)

clarification_plan = ActionPlan(
    request_type=RequestType.CLARIFICATION,
    needs_clarification=True,
    clarification_question="What would you like to do with your order?",
    reason="The intended action is unclear.",
)

cases = [
    ("Existing order", tracking_plan, {"order_id": "ORD-123"}, True),
    ("Unknown order", tracking_plan, {"order_id": "ORD-999"}, False),
    ("Missing ID", tracking_plan, {}, False),
    ("Blank ID", tracking_plan, {"order_id": "   "}, False),
    ("Invalid ID type", tracking_plan, {"order_id": 123}, False),
    ("Clarification", clarification_plan, {"order_id": "ORD-123"}, False),
]

for name, plan, arguments, expected_success in cases:
    result = executor.execute(plan, **arguments)

    assert result.success == expected_success, (name, result)
    print(f"{name}: {result.model_dump()}")

Existing order: {'success': True, 'action': 'track_order', 'data': {'order_id': 'ORD-123', 'status': 'shipped', 'estimated_delivery': '2026-09-23'}, 'message': None}
Unknown order: {'success': False, 'action': 'track_order', 'data': {}, 'message': 'Order ORD-999 was not found.'}
Missing ID: {'success': False, 'action': 'track_order', 'data': {}, 'message': 'order_id is required.'}
Blank ID: {'success': False, 'action': 'track_order', 'data': {}, 'message': 'order_id is required.'}
Invalid ID type: {'success': False, 'action': 'track_order', 'data': {}, 'message': 'order_id must be a string.'}
Clarification: {'success': False, 'action': 'none', 'data': {}, 'message': 'Cannot execute request type clarification.'}


In [1]:
# Continue testing
from agent1.analysis.action_plan import ActionPlan, RequestType
from agent1.conversation.state import ConversationState
from agent1.execution.action_executor import ActionExecutor
from agent1.routing.action_types import ActionType

state = ConversationState()
executor = ActionExecutor()

# First turn
state.add_user_message("Where is my order?")

# Simulate the resolved output of our analyzer.
plan = ActionPlan(
    request_type=RequestType.SINGLE_ACTION,
    intent="track_order",
    action=ActionType.ORDER_SERVICE,
    reason="The user wants to track an order.",
)

state.start_pending_action(
    action_plan=plan,
    required_arguments=["order_id"],
)

assert state.get_missing_arguments() == ["order_id"]

question = "What is your order number?"
state.add_assistant_message(question)

print("Agent:", question)

Agent: What is your order number?


In [2]:
# Second turn
user_reply = "ORD-123"
state.add_user_message(user_reply)

# For this example, the user supplies only the order ID.
state.update_pending_arguments(order_id=user_reply.strip())

assert state.get_missing_arguments() == []
assert state.pending_action_plan is not None

result = executor.execute(
    state.pending_action_plan,
    **state.pending_arguments,
)

state.add_tool_result(
    content=result.model_dump_json(),
)

# A simple response for this first working example.
if result.success:
    reply = (
        f"Your order {result.data['order_id']} "
        f"has status: {result.data['status']}."
    )
else:
    reply = result.message or "The order lookup failed."

state.add_assistant_message(reply)

# This execution attempt is finished; retain its history.
state.clear_pending_action()

print("Agent:", reply)

assert not state.has_pending_action()
assert len(state.get_messages()) == 5

Agent: Your order ORD-123 has status: shipped.


In [1]:
# Check instructions
from agent1.analysis.action_plan import ActionPlan, RequestType
from agent1.execution.argument_collector import ArgumentCollector
from agent1.routing.action_types import ActionType

collector = ArgumentCollector()

plan = ActionPlan(
    request_type=RequestType.SINGLE_ACTION,
    intent="track_order",
    action=ActionType.ORDER_SERVICE,
    reason="The user wants to track an order.",
)

examples = [
    ("ORD-123", {"order_id": "ORD-123"}),
    ("My order is ord-123.", {"order_id": "ORD-123"}),
    ("Please track ORD-456.", {"order_id": "ORD-456"}),
    ("I cannot find my order number.", {}),
    ("ORD-123 or ORD-456", {}),
]

for text, expected in examples:
    arguments = collector.extract(text, plan)
    assert arguments == expected, (text, arguments)
    print(text, "→", arguments)

ORD-123 → {'order_id': 'ORD-123'}
My order is ord-123. → {'order_id': 'ORD-123'}
Please track ORD-456. → {'order_id': 'ORD-456'}
I cannot find my order number. → {}
ORD-123 or ORD-456 → {}


In [2]:
# connect to conversation state
from agent1.conversation.state import ConversationState
from agent1.execution.action_executor import ActionExecutor

state = ConversationState()
executor = ActionExecutor()

# First turn
first_message = "Where is my order?"
state.add_user_message(first_message)

state.start_pending_action(
    action_plan=plan,
    required_arguments=collector.required_arguments(plan),
    arguments=collector.extract(first_message, plan),
)

if state.get_missing_arguments():
    question = "Please provide one order number, such as ORD-123."
    state.add_assistant_message(question)
    print("Agent:", question)

Agent: Please provide one order number, such as ORD-123.


In [3]:
user_reply = "My order is ORD-123."
state.add_user_message(user_reply)

pending_plan = state.pending_action_plan

if pending_plan is None:
    raise ValueError("There is no pending action.")

arguments = collector.extract(user_reply, pending_plan)
state.update_pending_arguments(**arguments)

if state.get_missing_arguments():
    reply = "Please provide one order number, such as ORD-123."
else:
    result = executor.execute(
        pending_plan,
        **state.pending_arguments,
    )

    state.add_tool_result(content=result.model_dump_json())

    if result.success:
        reply = (
            f"Your order {result.data['order_id']} "
            f"has status: {result.data['status']}."
        )
    else:
        reply = result.message or "The order lookup failed."

    state.clear_pending_action()

state.add_assistant_message(reply)
print("Agent:", reply)

Agent: Your order ORD-123 has status: shipped.


In [ ]:

# Testing of the ActionRunner with a simulated conversation
from agent1.analysis.action_plan import ActionPlan, RequestType
from agent1.conversation.state import ConversationState
from agent1.execution.action_runner import ActionRunner
from agent1.routing.action_types import ActionType

state = ConversationState()
runner = ActionRunner(state)

# Still simulating the analyzer's output for this focused check.
plan = ActionPlan(
    request_type=RequestType.SINGLE_ACTION,
    intent="track_order",
    action=ActionType.ORDER_SERVICE,
    reason="The user wants to track an order.",
)

first_message = "Where is my order?"
state.add_user_message(first_message)
print("Agent:", runner.start(plan, first_message))

follow_up = "My order is ORD-123."
state.add_user_message(follow_up)
print("Agent:", runner.resume(follow_up))

assert not state.has_pending_action()
assert len(state.get_messages()) == 5
